In [ ]:
import pandas as pd
import json

X = pd.read_parquet("/context/data/X.parquet")
y = pd.read_parquet("/context/data/y.parquet")

with open("/context/data/moons_split.json") as f:
    splits = json.load(f)

X_train      = X[X["moon"].isin(splits["train"])]
y_train      = y[y["moon"].isin(splits["train"])]
X_test_cloud = X[X["moon"].isin(splits["reduced_cloud"])]
y_test_cloud = y[y["moon"].isin(splits["reduced_cloud"])]


In [ ]:
import joblib
import os
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor


def train(X_train, y_train, model_directory_path, loop_moon, embargo):
    feature_cols = [c for c in X_train.columns if c.startswith("Feature_")]

    df = X_train.merge(y_train[["moon", "id", "target"]], on=["moon", "id"])
    df = df.dropna(subset=["target"])

    # ── ONLY CHANGE: normal time split — use first 80% of moons ──────────────
    all_moons   = sorted(df["moon"].unique())
    cutoff_idx  = int(len(all_moons) * 0.80)
    train_moons = all_moons[:cutoff_idx]
    held_moons  = all_moons[cutoff_idx:]

    df = df[df["moon"].isin(train_moons)]
    print(f"  Time split 80/20:")
    print(f"    Train : {len(train_moons)} moons ({train_moons[0]}→{train_moons[-1]})")
    print(f"    Held  : {len(held_moons)} moons  ({held_moons[0]}→{held_moons[-1]})")
    print(f"    Rows  : {len(df):,}")
    # ── END OF CHANGE ─────────────────────────────────────────────────────────

    X = df[feature_cols].fillna(0)
    y = df["target"]

    model = LGBMRegressor(
        n_estimators=200,
        learning_rate=0.05,
        random_state=42
    )

    model.fit(X, y)

    joblib.dump(model, os.path.join(model_directory_path, "model.joblib"))

In [ ]:
def infer(X_test, model_directory_path, loop_moon, embargo):
    model        = joblib.load(os.path.join(model_directory_path, "model.joblib"))
    feature_cols = [c for c in X_test.columns if c.startswith("Feature_")]

    preds = model.predict(X_test[feature_cols].fillna(0))

    return pd.DataFrame({
        "moon":       X_test["moon"].values,
        "id":         X_test["id"].values,
        "prediction": preds,
    })

In [ ]:
import os
import numpy as np
from scipy.stats import pearsonr

embargo   = 4
model_dir = "./model"

os.makedirs(model_dir, exist_ok=True)

train(X_train, y_train, model_dir, loop_moon=0, embargo=embargo)

results = []

for moon in splits["reduced_cloud"]:
    pred = infer(
        X_test_cloud[X_test_cloud["moon"] == moon],
        model_dir,
        loop_moon=moon,
        embargo=embargo
    )
    results.append(pred)

predictions = pd.concat(results)

corrs = []

for moon in splits["reduced_cloud"]:
    merged = predictions[predictions["moon"] == moon].merge(
        y_test_cloud[y_test_cloud["moon"] == moon],
        on=["moon", "id"]
    )

    if len(merged) > 1:
        r, _ = pearsonr(merged["prediction"], merged["target"])
        corrs.append(r)
        print(f"Moon {moon}: Pearson r = {r:.4f}")

print(f"\nMean IC = {np.mean(corrs):.4f}")